[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbutler2405/EMP5027-rows-to-pixels/blob/main/notebooks/practical-2-data-structures-functions-modules/EMP5027-Lecture-2-Data-Structures-Functions-Modules.ipynb)


# EMP5027 - Methods in Data Analysis & Quality Assurance
## Lecture 2: Data Structures, Functions, and Modules

**Instructor:** Dr. Liam Butler · University of Malta

Last time we got Python running and pushed some variables and lists around. Today we build on that
and start writing code the way you will actually need to for your research: organised into
reusable pieces, with data held in the structure that actually fits the problem, rather than
whatever happens to work first. By the end of this notebook you should be comfortable choosing
between lists, tuples, sets, dicts, NumPy arrays and pandas DataFrames, and you should be able to
write your own small functions and even your own module, so that the logic you use in one notebook
does not have to be retyped in the next one.


## Learning Objectives
By the end of this lecture, you should be able to:
- Explain when to use lists, tuples, sets, dicts, NumPy arrays, and pandas DataFrames, and justify the choice for a given environmental dataset.
- Import and use standard library, third-party, and your own modules.
- Write functions with clear docstrings and type hints (including units, ranges, and limits, which matter a great deal in environmental science).
- Reshape data between long and wide layouts, compute abundance and richness, and perform simple QA/QC flags.
- Work with time series, missing data, and simple raster concepts using NumPy arrays.


## Quick Recap (Lecture 1) & Today
**Previously:** we covered variables, lists and dicts, slicing, strings, basic file I/O, and quick
descriptive statistics. If any of that still feels shaky, it is worth a quick re-read of the
Lecture 1 notebook before we push on, because everything today builds directly on it.

**Today:** we will organise a small project properly, write and import our own module, go deeper
into the core data structures, write functions that support reproducible QA/QC, move between tidy
(long) and wide data layouts, and finish with the basic logic behind raster data, which you will
meet again and again once we get into remote sensing.


## Organise Your Project Early
Once a project grows past a single notebook, a clean folder layout is what keeps it reproducible,
lets a collaborator (or your future self, six months from now) find things, and makes it possible
to actually audit what was done. A typical structure looks like this:

```
project/
├─ data/                # raw/processed datasets
├─ notebooks/
│  └─ 02_lecture2_demo.ipynb
├─ src/
│  ├─ helpers.py        # QA checks, units, shared utilities
│  └─ species.py        # biodiversity helpers (e.g., richness)
└─ environment.yml      # optional: conda env spec
```

The habit to build now, even for small class exercises, is to keep reusable logic out of the
notebook and in a plain `.py` file under `src/`, then import it. That is exactly what we will do
in a few cells' time with `helpers.py`.


## Modules: Standard, Third-Party, and Your Own
Python code you can `import` falls into three categories, and it helps to know which is which:
- **Standard library:** ships with Python itself, no installation needed (e.g. `math`, `pathlib`, `datetime`, `statistics`).
- **Third-party:** written by other people or teams, installed separately with conda or pip (e.g. `numpy`, `pandas`, `matplotlib`, `scikit-learn`).
- **Your own:** plain Python files you write yourself (e.g. `helpers.py`), which you can then `import` just like any other module once it is saved next to your notebook or on your Python path.


## Running this in Google Colab

Click the badge above to open this notebook directly in Colab, no local setup required.
Everything this notebook needs is already available on Colab by default.

In [ ]:
# --- Google Colab setup (safe to run locally too, it just skips this step) ---
import sys  # gives us access to sys.modules, which lists every module currently imported

# "google.colab" only ever appears in sys.modules when we are actually running inside Colab,
# so checking for it is a reliable way to detect the environment without any extra packages
if "google.colab" in sys.modules:
    pass  # nothing to install here, Colab already has everything this notebook needs
    print("Running in Colab, ready to go.")
else:
    # running locally (e.g. on your own laptop via Jupyter), so we just trust that you
    # followed the setup instructions from Practical 1 and have the packages installed already
    print("Not running in Colab, assuming packages are already installed locally.")


In [ ]:
import math                     # standard library: general maths functions
import statistics as stats      # stdlib with alias, gives us mean/median/etc without NumPy
import numpy as np              # third-party: fast numeric arrays, imported under its usual alias

print(math.sqrt(16))            # 4.0, a plain scalar square root
print(stats.mean([1, 2, 3]))    # 2, the mean of a short list using pure-Python statistics
print((np.array([1, 2, 3]) * 2).tolist())  # [2, 4, 6], NumPy applies * 2 to every element at once


## Import Patterns & Namespaces
Prefer explicit imports and the community-standard aliases everyone else in the field uses,
so your code is instantly readable to a collaborator:
- `import numpy as np`, `import pandas as pd`, `import matplotlib.pyplot as plt`
- Avoid `from module import *`. It dumps every name from that module into your notebook's
  namespace, which hides where a function actually came from and risks silently overwriting
  a name you already had. It looks like it saves typing, but it costs you clarity later.


In [ ]:
import numpy as np
import pandas as pd
from statistics import median   # a targeted import: we only need this one function from statistics

print(median([9, 1, 3]))        # 3, the middle value once sorted
a = np.array([1, 2, 3])
print(a.dtype, a.shape)         # dtype tells us the element type, shape tells us the array's dimensions
df = pd.DataFrame({"x": [1,2], "y": [3,4]})  # a tiny two-column table
print(df.head())                # .head() previews the first rows, here all of them since it's so small


## Writing Your Own Module: `helpers.py`
Rather than copy-pasting the same QA/QC functions into every notebook you write this semester,
we will define them once in a file called `helpers.py` and import that file wherever we need it.
The cell below actually creates `helpers.py` in the current folder by writing text to disk, so
after running it you will have a real Python file sitting alongside this notebook that you could
open and edit directly.


In [ ]:
# Write a simple helpers.py file
# (outer wrapper uses '''...''' since the functions' own docstrings use """...""", and
# using the same delimiter for both would close the outer string early)
helpers_src = '''
def c_to_k(c: float) -> float:
    """Convert Celsius (°C) to Kelvin (K)."""
    return c + 273.15

def exceed(val: float, limit: float = 50.0) -> bool:
    """Return True if 'val' exceeds the regulatory 'limit'."""
    return val > limit

def flag_range(val: float, lo: float, hi: float) -> bool:
    """QA/QC: True if outside plausible physical range [lo, hi]."""
    return not (lo <= val <= hi)
'''

# open the file in write mode ("w") and use utf-8 explicitly so the ° symbol saves correctly
with open("helpers.py", "w", encoding="utf-8") as f:
    _ = f.write(helpers_src)  # the underscore just discards the character count write() returns

print("helpers.py written.")


In [ ]:
import importlib, helpers       # helpers.py is importable now because it lives in this folder
importlib.reload(helpers)       # forces Python to re-read helpers.py, useful if we edit it and rerun

print(helpers.c_to_k(25.0))          # 298.15, 25 degrees C converted to Kelvin
print(helpers.exceed(55))            # True, 55 exceeds the default limit of 50
print(helpers.flag_range(999, 0, 200))  # True, 999 is well outside the plausible range [0, 200]


## Standard Library Mini-Tour
A handful of standard library modules turn up constantly in environmental data work, so it is
worth knowing them by name: `pathlib` for building file paths that work on any operating system,
`datetime` for handling timestamps and date arithmetic, and `collections.Counter` for quickly
tallying how often each value occurs in a list.


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter

print("Project root:", Path('.').resolve())  # resolve() turns "." into a full, absolute path
# timedelta lets us do date arithmetic directly, here adding 7 days to right now
print("Next week:", (datetime.now() + timedelta(days=7)).strftime("%Y-%m-%d"))
# Counter builds a dict-like tally of how many times each item appears in the list
print("Counter:", Counter(["OK","OK","EXC","OK"]))


## Choosing the Right Data Structure
Every dataset you meet in this module will fit one of these shapes better than the others, and
picking the right one up front will save you a lot of awkward workarounds later:
- **List `[ ]`**: ordered, flexible, and can change size (e.g. daily rainfall readings for one site).
- **Tuple `( )`**: ordered, but immutable once created (e.g. a fixed `(lat, lon)` coordinate pair that should never accidentally be edited).
- **Set `{ }`**: an unordered collection of unique elements, useful for deduplicating (e.g. a set of species names with no repeats).
- **Dict `{k: v}`**: labelled lookups, mapping a key to a value (e.g. site name to its measured value(s)).
- **NumPy array**: numeric data with fast, vectorised maths (e.g. a sensor's time series of readings).
- **pandas DataFrame**: labelled tables with rows and columns, ideal for time series, joins, and export to CSV.


## Lists: Filter & Transform (Field Logistics)
A very common task in the field is starting from a full list of sites and narrowing it down, or
turning it into something printable for a field sheet. Both of those are list operations, so we
will practise them here with a small list comprehension for each.


In [ ]:
sites = ["VAL01", "GOZ02", "MAR03", "MEL04"]

# Filter: choose "coastal" sites (toy rule here: site code ends in 03 or 04)
coastal = [s for s in sites if s.endswith("03") or s.endswith("04")]

# Transform: build a printable instruction label for each site, for a field sheet
labels = [f"Visit site {s}" for s in sites]

print("Coastal sites:", coastal, "| N =", len(coastal))
print("Labels:", labels)


## Tuples (Fixed Records) & Sets (Unique Species)
Tuples are the right choice when a value genuinely should not change once it is created, such as
a station's coordinates. Sets are the right choice whenever we only care about which distinct
values are present, such as which species were observed, regardless of how many times each one
turned up.


In [ ]:
# Tuple demo via dict: station -> (lat, lon) as a fixed, immutable coordinate pair
coords = {"VAL01": (35.9, 14.5), "GOZ02": (36.0, 14.3)}
print("VAL01 coords:", coords["VAL01"])

# Sets automatically drop duplicates, which is exactly what we want for a species list
observed = ["eel", "trout", "eel", "sturgeon"]
unique_species = set(observed)
print("Unique species:", sorted(unique_species))  # sorted() turns the set into a readable, ordered list
print("Species richness:", len(unique_species))    # richness is simply the count of unique species


## Dictionaries as Mini-LIMS (Lookup & Flags)
A Laboratory Information Management System (LIMS) is, at its core, just a set of lookups: given a
sample, what were its measured parameters? A nested dictionary mirrors that structure directly,
with the outer key being the site and the inner dictionary holding each parameter's value.


In [ ]:
wq = {
    "VAL01": {"NO3": 32, "PO4": 0.8},
    "GOZ02": {"NO3": 55, "PO4": 1.1},
    "MAR03": {"NO3": 41, "PO4": 0.9},
}

LIMIT_NO3 = 50  # mg/L (illustrative regulatory limit)
# .items() lets us loop over key and value together, here site name and its parameter dict
for site, params in wq.items():
    no3 = params["NO3"]
    if no3 > LIMIT_NO3:
        print(site, "→ NO3 EXCEEDS")
    else:
        print(site, "→ OK")


## From Dict-of-Dicts to pandas DataFrame
A nested dictionary is fine for small, one-off lookups, but a DataFrame gives us far more once the
dataset grows: easy summaries, filtering, export to CSV, and plotting. Converting between the two
is straightforward, so there is no reason to stay stuck in dictionary form once you need any of
that.


In [ ]:
import pandas as pd

wq = {
    "VAL01": {"NO3": 32, "PO4": 0.8},
    "GOZ02": {"NO3": 55, "PO4": 1.1},
    "MAR03": {"NO3": 41, "PO4": 1.5},
}

# orient="index" tells pandas that the outer dict keys (site names) should become the row index
df = pd.DataFrame.from_dict(wq, orient="index")
print("DataFrame:\n", df)

LIMITS = {"NO3": 50, "PO4": 1.0}
# loop over each parameter's limit and add a new boolean column flagging any exceedance
for param, limit in LIMITS.items():
    df[param + "_exceed"] = df[param] > limit

print("\nWith flags:\n", df)
print("\nMeans:\n", df[["NO3","PO4"]].mean())


## NumPy Arrays: Sensor Time Series & Masks
Sensor data is naturally numeric and often arrives as a long sequence of readings, exactly the
case NumPy arrays are built for. Here we simulate a day of PM2.5 (fine particulate matter)
readings and use a boolean mask to count how many hours exceeded a threshold.


In [ ]:
import numpy as np

hrs   = np.arange(24)                              # hours 0 to 23
base  = 22 + 9*np.sin(2*np.pi*hrs/24)              # a smooth daily cycle in the baseline PM2.5 level
rng   = np.random.default_rng(42)                  # a seeded random generator, so results are reproducible
noise = rng.normal(0, 4, 24)                       # random variation added on top of the daily cycle
pm25  = np.clip(base + noise, 0, None)             # clip negative values to 0, since PM2.5 can't be negative

exceed_mask = pm25 > 35                            # a boolean array, True where the WHO-style threshold is exceeded
print("Exceedance hours:", int(exceed_mask.sum()), "/", pm25.size)  # sum() on booleans counts the True values


## pandas DataFrames: Tabular + Time Index
Once data has a natural time order, a pandas DatetimeIndex lets us treat the DataFrame's row
labels as actual dates, which unlocks convenient time-based operations like resampling, which
we will use shortly.


In [ ]:
import pandas as pd

days = pd.date_range("2025-02-01", periods=7, freq="D")  # 7 consecutive daily timestamps
df = pd.DataFrame({"NO2": [18, 22, 20, 25, 21, 17, 16]}, index=days)  # the dates become the row index

print(df.head(3))                       # first 3 rows only
print("Mean NO2:", float(df["NO2"].mean()))


## Air Quality Case: Daily Means & Flags
A realistic QA/QC task: given hourly PM2.5 readings for a week, aggregate them to daily means and
flag any day whose mean exceeds a regulatory threshold (here, 35 microgrammes per cubic metre).


In [ ]:
import numpy as np, pandas as pd

hours = pd.date_range("2025-03-01", periods=24*7, freq="H")   # one timestamp per hour, for 7 days
rng   = np.random.default_rng(0)
base  = 23 + 8*np.sin(2*np.pi*hours.hour/24)                  # daily cycle keyed to the hour of day
noise = rng.normal(0, 4, len(hours))
pm25  = np.clip(base + noise, 0, None)                        # again, no negative concentrations

aq = pd.DataFrame({"PM25": pm25}, index=hours)
DAILY_LIMIT = 35.0
daily = aq.resample("D").mean(numeric_only=True)              # collapse hourly rows down to one mean per day
daily["exceed_35"] = daily["PM25"] > DAILY_LIMIT               # flag any day above the limit
print(daily)


## Missing Data (Detect & Simple Fill)
Real sensor and lab data almost always has gaps, from a sensor dropout, a missed sample, or a
failed QA check. The first step is always to count how much is missing before deciding what to
do about it. Here we fill with the median, which is a simple, transparent, and reproducible
choice for a short demonstration, though the right choice for your own data will depend on why
the values are missing.


In [ ]:
import pandas as pd
s = pd.Series([22, 24, None, 23, None, 25, 26])
missing_n = int(s.isna().sum())          # isna() flags missing values, sum() counts how many
filled = s.fillna(s.median())            # fill each missing value with the series' own median
print("Missing count:", missing_n)
print("Filled series:\n", filled)


## Functions for Structure, QA, and Reuse
Once the same calculation shows up more than once, it belongs in a function. Small, focused
functions are easier to read, easier to test on their own, and much easier to audit later when
someone (possibly you) asks exactly how a number was calculated.


In [ ]:
def daily_mean(values: list[float]) -> float:
    """Mean of a list of numbers (e.g., daily pollutant values)."""
    return sum(values) / len(values)

print(daily_mean([1, 2, 3, 4]))


## Parameters, Defaults, and Named Arguments
Giving a parameter a default value means callers only need to specify it when they want something
different from the usual case, which keeps most calls short. Passing arguments by name, rather
than relying on position alone, also makes the call itself self-documenting.


In [ ]:
def exceed(val: float, limit: float = 50.0) -> bool:
    """True if 'val' exceeds 'limit'."""
    return val > limit

print(exceed(55))              # uses the default limit of 50, so this is True
print(exceed(55, limit=60))    # limit overridden by name, so this is False


## QA/QC Flag Function (Reusable)
A single reading often needs more than one check at once: is it physically plausible, and does it
breach a regulatory limit? Returning both flags from one function keeps the two questions together
and avoids calling separate functions with the same value twice.


In [ ]:
def qa_qc_flag(val: float, lo: float, hi: float, limit: float) -> dict:
    """Return QA/QC flags for one measurement."""
    return {
        "phys_out_of_range": not (lo <= val <= hi),   # True if val falls outside the plausible physical range
        "exceeds_reg_limit": val > limit               # True if val breaches the regulatory limit
    }

vals = [32, 55, -5, 999]
for v in vals:
    print(v, ":", qa_qc_flag(v, lo=0, hi=200, limit=50))


## Returning Multiple Values (Tuples)
A function can return more than one value at once by packing them into a tuple, and the caller can
unpack them straight into separate variables on the receiving line, which reads very naturally.


In [ ]:
def summary_stats(values: list[float]) -> tuple[float, float, float]:
    mean = sum(values) / len(values)
    return mean, min(values), max(values)   # returned together as a tuple of three values

m, lo, hi = summary_stats([5, 7, 9, 1])      # unpacked directly into three named variables
print("mean:", m, "| min:", lo, "| max:", hi)


## Higher-Order Functions
A function that accepts another function as an argument is called a higher-order function. This
lets us write one general-purpose pipeline and reuse it with whatever transformation we need at
the time, rather than writing a near-identical function for every transformation.


In [ ]:
def apply_and_mean(values: list[float], fn) -> float:
    transformed = [fn(v) for v in values]     # apply fn to every value first
    return sum(transformed) / len(transformed)  # then take the mean of the transformed values

print(apply_and_mean([1, 2, 3], lambda x: x * x))       # mean of the squares: 1, 4, 9
print(apply_and_mean([2.5, 3.0, 4.0], lambda x: x * 1000))  # mg/L converted to µg/L, then averaged


## Long ↔ Wide (Pivoting)
These are the two standard shapes tabular data takes. Long format has one observation per row,
which is what most grouping and statistical functions expect. Wide format has one row per site
with a separate column for each variable, which is what most people actually want to read in a
report. We will build a long table here and then pivot it to wide.


In [ ]:
import pandas as pd

wq = {
    "S1": {"NO3": [28, 30, 33], "PO4": [0.7, 0.8, 0.9]},
    "S2": {"NO3": [55, 59, 61], "PO4": [1.0, 1.2, 1.1]},
    "S3": {"NO3": [40, 42, 41], "PO4": [0.9, 0.9, 1.0]},
}

# Build a long table: one row per (site, parameter, value) observation
rows = []
for site, params in wq.items():
    for v in params["NO3"]:
        rows.append((site, "NO3", v))
    for v in params["PO4"]:
        rows.append((site, "PO4", v))

df_long = pd.DataFrame(rows, columns=["Site", "Param", "Value"])
print("Long format head:\n", df_long.head())

# Pivot to wide: one row per site, one column per parameter, aggregated by mean
pivot = df_long.pivot_table(index="Site", columns="Param", values="Value", aggfunc="mean")
pivot["NO3_exceed"] = pivot["NO3"] > 50
print("\nWide summary:\n", pivot)


## Using Real Data: Penguins (Abundance & Richness)
Time to move off toy data and onto a real dataset. We will use the penguins dataset that ships
with seaborn, and treat **island** as the "site" and **species** as the taxon, so that the
abundance and richness calculations mirror exactly what you would do with real field survey data.
> If `seaborn` is missing, install with `pip install seaborn` (or `conda install seaborn`).


In [ ]:
try:
    import seaborn as sns
except ImportError as e:
    raise ImportError("Seaborn is required to load the penguins dataset. Install with `pip install seaborn`.") from e

import pandas as pd

penguins = sns.load_dataset("penguins")   # loads the bundled penguins dataset as a DataFrame
print(penguins.head())

# Drop rows missing key fields, since we can't group by species or island if either is unknown
penguins = penguins.dropna(subset=["species", "island"])

# Abundance per island: how many individuals of each species were recorded on each island
abundance = (
    penguins.groupby(["island", "species"]).size().reset_index(name="count").sort_values(["island","species"])
)
print("\nAbundance (long):\n", abundance.head())

# Wide report view: one row per island, one column per species count, easier to read as a table
abundance_wide = abundance.pivot(index="island", columns="species", values="count").fillna(0).astype(int)
print("\nAbundance (wide):\n", abundance_wide)

# Richness per island: the number of distinct species present, regardless of how many individuals
richness = penguins.groupby("island")["species"].nunique().reset_index(name="richness").sort_values("island")
print("\nRichness per island:\n", richness)

# Cumulative species list across the whole dataset, sorted alphabetically for readability
cumulative_species = sorted(penguins["species"].dropna().unique().tolist())
print("\nCumulative species:", cumulative_species, "| N =", len(cumulative_species))


## Dictionary Views & Merges
We can also build the island-to-species relationship directly as a dictionary of sets, which is a
handy structure when you want to quickly check membership (is this species on this island?)
rather than produce a table. We then bring abundance and richness back together into a single
summary DataFrame using merges, the same operation you would use to combine any two related
tables.


In [ ]:
# For each island, collect the set of species observed there (a dict mapping island -> set of species)
island_to_species = (
    penguins.groupby("island")["species"]
    .apply(lambda s: set(s.dropna()))
    .to_dict()
)

# Total number of individuals recorded per island, as a plain dict
abundance_by_island = penguins.groupby("island")["species"].size().to_dict()

for isl, spp in island_to_species.items():
    abundance_total = abundance_by_island[isl]
    richness = len(spp)   # richness is just how many distinct species are in the set
    print(f"{isl}: abundance={abundance_total} | richness={richness} | species={sorted(spp)}")

# Merge three separate summaries (abundance, richness, species list) into one DataFrame
abundance_per_island = abundance.groupby("island")["count"].sum().reset_index(name="abundance")
richness_per_island  = abundance.groupby("island")["species"].nunique().reset_index(name="richness")
species_list_per_island = (
    abundance.groupby("island")["species"]
    .unique()
    .apply(lambda spp: sorted(spp))
    .reset_index(name="species_list")
)

# merge() joins tables on a shared column, here "island", the same way you would join database tables
summary = abundance_per_island.merge(richness_per_island, on="island").merge(species_list_per_island, on="island")
print("\nIsland summary:\n", summary)


## Raster Concepts (2D Grids) with NumPy
So far every dataset has been a table, one row per observation. A great deal of environmental
data, especially anything from satellites or interpolated climate surfaces, instead comes as a
raster: a regular grid of values. We will build some small synthetic rasters here so you can see
the underlying logic before you meet the real thing in a later lecture on remote sensing.


## Introduction to Rasters
- A raster is a grid of cells (pixels), with each cell storing a single value.
- Think of it as a 2D spreadsheet of environmental values, where row and column position stands in for location.
- It is the dominant format for remote sensing imagery and for climate or ecology monitoring products, so it is worth being comfortable with it early.


In [ ]:
import numpy as np
np.random.seed(1)  # fixed seed so this synthetic raster is reproducible across runs
temp = 18 + 7 * np.random.rand(10, 10)  # 10×10 temperature raster (°C), random values between 18 and 25
print("Raster shape:", temp.shape)      # (rows, columns), here 10 by 10
print("Example cell:", float(temp[0,0]))  # the value stored at row 0, column 0


## Classifying Thermal Hotspots
A very common raster operation is thresholding: turning a continuous grid of values into a
boolean grid marking which cells meet some condition, here any cell warmer than 22.5 degrees C.


In [ ]:
threshold = 22.5
hot = temp > threshold                     # a boolean 10x10 array, True where temp exceeds the threshold
print("Hot cells:", int(hot.sum()), "of", temp.size)  # count of hot cells out of the total cell count


## Adding a Second Raster & Overlays
Rasters can be combined cell by cell, as long as they share the same shape. This is exactly how
you would overlay, say, a temperature raster and a chlorophyll raster to find locations where both
conditions hold at once, which is what we do below.


In [ ]:
chl = 1.5 + 0.8 * np.random.rand(10, 10)   # a second synthetic raster: chlorophyll (mg/m³)
hot = temp > 22.5
eutrophic = chl > 2.0
hot_eutro = hot & eutrophic                # element-wise AND: True only where both conditions hold
print("Hot cells:", int(hot.sum()))
print("Eutrophic cells:", int(eutrophic.sum()))
print("Hot + Eutrophic cells:", int(hot_eutro.sum()))


## Simple Raster Statistics & Correlation
Once we have a boolean mask, we can use it to index directly into another raster, letting us
compare chlorophyll values inside versus outside the hotspots. We can also flatten both rasters
and compute a correlation coefficient to check whether the two variables tend to move together
across the whole grid.


In [ ]:
chl_hot = chl[hot]     # boolean indexing: keep only the chlorophyll values where hot is True
chl_not = chl[~hot]    # ~hot inverts the mask, so this keeps values where hot is False
print("Mean chlorophyll in hotspots:", float(chl_hot.mean()))
print("Mean chlorophyll outside hotspots:", float(chl_not.mean()))

import numpy as np
# ravel() flattens each 2D raster into a 1D array so corrcoef can compare them cell for cell
r = float(np.corrcoef(temp.ravel(), chl.ravel())[0,1])
print("Correlation (temp vs chl):", r)


## Visualising Rasters & Scatter
A raster is easiest to interpret as an image, and the relationship between two rasters is easiest
to check with a scatter plot. Each chart below uses its own figure with default styling, we will
cover customising colours, colour maps and layouts properly in a later lecture.


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.imshow(temp)           # displays the 2D array as an image, using the default colormap
plt.title("Temperature (°C)")
plt.colorbar()             # adds a scale showing which colour corresponds to which value
plt.show()

plt.figure()
plt.imshow(chl)            # default colormap
plt.title("Chlorophyll (mg/m³)")
plt.colorbar()
plt.show()

plt.figure()
plt.scatter(temp.ravel(), chl.ravel())   # flatten both rasters so each cell becomes one point
plt.xlabel("Temperature (°C)")
plt.ylabel("Chlorophyll (mg/m³)")
plt.title("Temp vs Chlorophyll")
plt.show()


## Common Pitfalls & How to Avoid Them
A quick reference for mistakes that come up often enough to be worth naming directly.

| Pitfall                    | Solution                                                  |
|---------------------------|-----------------------------------------------------------|
| Mixing types in arrays    | Use **NumPy arrays** for numeric work                     |
| Copy-pasting code blocks  | Write **functions** with docstrings & tests               |
| Mystery imports           | Use **explicit imports** (e.g., `import numpy as np`)     |
| Spaghetti notebooks       | Move logic into **modules** (e.g., `helpers.py`)         |
| Silent missing data       | **Count** missing, **log** imputation decisions            |
| No documentation          | Add **type hints** & **docstrings** (units, ranges)       |
| Hard-coded paths          | Use **pathlib** for portable paths                        |
| Not fixing random seeds   | Set **seeds** (e.g., `np.random.seed`) for reproducible runs |
| Messy project structure   | Keep `/data`, `/notebooks`, `/src`                        |
| Overcomplicating analysis | Prefer **simpler models** if they work                    |


## Practice Exercises
Work through these in order, they build on the functions and structures used above.
1. Create `helpers.py` functions for unit conversions (e.g. mg/L to µg/L), each with a docstring and type hints.
2. Build a dict-of-dicts for three sites with NO3 and PO4, convert it to a DataFrame, flag exceedances, and compute means.
3. Create a 7-day hourly PM2.5 time series, compute daily means and a daily `exceed_35` column.
4. Make synthetic rasters (10×10) for temperature and chlorophyll, compute hot and eutrophic cell counts and the correlation between them.
5. Using `penguins`, produce a richness bar chart per island (one figure, default styles), and a table listing the species list per island.
6. (Stretch) Write a higher-order function that takes a list of values and a list of functions, applying each one in sequence as a mini pipeline.


## Summary & What's Next
- Import smartly, and write your own modules once logic needs to be reused across notebooks.
- Choose data structures that mirror the science: lists, tuples, sets and dicts for small, flexible collections, arrays and DataFrames once the data becomes numeric or tabular at scale.
- Package repeated steps into functions with clear docstrings and type hints.
- We applied all of this to air quality, water quality, species records, and raster grids.

**Up next:** structured data at scale (CSV and Parquet), joins and merges, plotting patterns and QA/QC, our first statistical models, and a short introduction to SQL.
